In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install codecarbon

In [ ]:
pip install carbontracker

Import all libraries

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from codecarbon import EmissionsTracker
except ImportError as exc:
    raise ImportError(
        "CodeCarbon bulunamadı. Lütfen önce şunu çalıştır: pip install codecarbon"
    ) from exc

# Reading the data

In [ ]:
# 1. Add the file paths (URLs) into a list in order
urls = [
    "/content/drive/MyDrive/Green Computing/five_EHRs_public_datasets/10_7717_peerj_5665_dataYM2018_neuroblastoma.csv",
    "/content/drive/MyDrive/Green Computing/five_EHRs_public_datasets/dataset_Belgrade2021_pediatric_brain_tumor_plos_one_0259095_cleaned.csv",
    "/content/drive/MyDrive/Green Computing/five_EHRs_public_datasets/dataset_Taipei2018_colorectal_cancer_EHRs_plos_one_0200893_final_cleaned.csv",
    "/content/drive/MyDrive/Green Computing/five_EHRs_public_datasets/journal.pone.0148699_S1_Text_Sepsis_SIRS_EDITED.csv",
    "/content/drive/MyDrive/Green Computing/five_EHRs_public_datasets/journal.pone.0158570_S2File_depression_heart_failure.csv"
]

# 2. Run the loop in one line and unpack the 5 results into 5 separate variable names:
neuro, braint, cancer, sepsis, depres = [pd.read_csv(url) for url in urls]


# Desriptive analysis


In [ ]:
sepsis.shape

(1257, 16)

In [ ]:
cancer.head(30)

,bian_ma,age,gender,asa,asa3,dm,cad,hf,cva,ckd,...,signet_ring,lymphovascularinvasion,perineural,ct,rt,nactrt,interval,progress,interval_r,death
0,2,52,1,3,1,0,0,0,0,0,...,0,1,0,1,0,0,11.761807,1,9.856263,1
1,21,85,2,2,0,0,1,0,0,0,...,0,0,0,1,0,0,8.377823,1,7.063655,1
2,22,45,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,4.763860,1,2.792608,1
3,23,57,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,2.595483,1,1.774127,1
4,29,43,1,2,0,1,0,0,0,0,...,0,0,0,1,0,0,16.164271,1,7.096509,1
5,33,51,2,2,0,0,0,0,0,0,...,0,0,0,1,0,0,22.767967,1,2.825462,0
6,38,64,2,2,0,0,0,0,0,0,...,9,9,9,1,0,0,16.361396,1,9.691992,0
7,43,71,1,3,1,0,0,0,0,0,...,0,0,0,1,0,0,12.878850,1,5.848049,1
8,47,82,1,2,0,0,0,0,0,0,...,9,9,9,1,0,0,4.402464,1,2.365503,1
9,48,81,1,2,0,0,0,0,0,0,...,0,1,0,1,1,0,9.166324,1,5.946612,0


In [ ]:
cancer.head(5)

,bian_ma,age,gender,asa,asa3,dm,cad,hf,cva,ckd,...,signet_ring,lymphovascularinvasion,perineural,ct,rt,nactrt,interval,progress,interval_r,death
0,2,52,1,3,1,0,0,0,0,0,...,0,1,0,1,0,0,11.761807,1,9.856263,1
1,21,85,2,2,0,0,1,0,0,0,...,0,0,0,1,0,0,8.377823,1,7.063655,1
2,22,45,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,4.763860,1,2.792608,1
3,23,57,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,2.595483,1,1.774127,1
4,29,43,1,2,0,1,0,0,0,0,...,0,0,0,1,0,0,16.164271,1,7.096509,1


In [ ]:
cancer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   bian_ma                 999 non-null    int64  
 1   age                     999 non-null    int64  
 2   gender                  999 non-null    int64  
 3   asa                     999 non-null    int64  
 4   asa3                    999 non-null    int64  
 5   dm                      999 non-null    int64  
 6   cad                     999 non-null    int64  
 7   hf                      999 non-null    int64  
 8   cva                     999 non-null    int64  
 9   ckd                     999 non-null    int64  
 10  cea                     999 non-null    float64
 11  log_cea                 999 non-null    float64
 12  laparoscopic            999 non-null    int64  
 13  tumor_loc               999 non-null    int64  
 14  ea                      999 non-null    in

In [ ]:
# ------------------------------------------------------------
# 0. Dataset-specific target column names
# ------------------------------------------------------------
# The target is no longer taken from the last column. The target name is fixed for each dataset here.
TARGET_COLUMN_MAP: Dict[str, str] = {
    "neuro": "outcome",
    "braint": "Status OS",
    "cancer": "death",
    "sepsis": "Mortality",
    "depres": "Death (1=Yes, 0=No)",
}


def get_target_column_name(df: pd.DataFrame, dataset_name: str) -> str:
    """Return the target column based on the dataset name; the last-column target assumption is not used."""
    if dataset_name not in TARGET_COLUMN_MAP:
        raise KeyError(
            f"Dataset '{dataset_name}' has no target definition in TARGET_COLUMN_MAP. "
            f"Available keys: {list(TARGET_COLUMN_MAP.keys())}"
        )

    target_name = TARGET_COLUMN_MAP[dataset_name]
    if target_name not in df.columns:
        raise ValueError(
            f"Dataset '{dataset_name}' target column not found for: '{target_name}'. "
            f"Available columns: {list(df.columns)}"
        )
    return target_name


# ------------------------------------------------------------
# 1. Load datasets from notebook global variables
# ------------------------------------------------------------
def get_dataframes_from_memory() -> Dict[str, pd.DataFrame]:
    """
    Loads preloaded DataFrames from the Colab/Jupyter environment.

    Expected DataFrame names:
    - neuro
    - braint
    - cancer
    - sepsis
    - depres
    """
    required_names = ["neuro", "braint", "cancer", "sepsis", "depres"]
    dataframes = {}

    for name in required_names:
        if name not in globals():
            raise NameError(
                f"'{name}' DataFrame not found. First read the CSV file and assign it to the {name} variable."
            )
        df = globals()[name]
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"'{name}' is not a pandas DataFrame.")
        dataframes[name] = df.copy()

    return dataframes


# ------------------------------------------------------------
# 2. General data cleaning functions
# ------------------------------------------------------------
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Convert column names to strings, strip leading/trailing whitespace, and make duplicate column names unique."""
    df = df.copy()
    cleaned = df.columns.astype(str).str.strip()

    seen = {}
    new_cols = []
    for col in cleaned:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    df.columns = new_cols
    return df


def convert_numeric_like_columns(X: pd.DataFrame, threshold: float = 0.90) -> pd.DataFrame:
    """
    Converts object/string columns that actually contain numeric values into numeric columns.
    A column is converted if at least 90% of its values can be converted to numeric.
    """
    X = X.copy()
    for col in X.columns:
        if pd.api.types.is_numeric_dtype(X[col]):
            continue

        converted = pd.to_numeric(X[col], errors="coerce")
        non_missing_original = X[col].notna().sum()
        if non_missing_original == 0:
            continue

        conversion_rate = converted.notna().sum() / non_missing_original
        if conversion_rate >= threshold:
            X[col] = converted
        else:
            X[col] = X[col].astype("object")
    return X


def remove_duplicate_feature_columns(X: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """Remove duplicate feature columns that contain exactly the same values."""
    duplicate_mask = X.T.duplicated()
    removed_cols = X.columns[duplicate_mask].tolist()
    X_clean = X.loc[:, ~duplicate_mask].copy()
    return X_clean, removed_cols


def get_col(X: pd.DataFrame, col: str) -> Optional[pd.Series]:
    """Return a numeric Series if the column exists; otherwise return None."""
    if col not in X.columns:
        return None
    return pd.to_numeric(X[col], errors="coerce")


def safe_divide(a: pd.Series, b: pd.Series, eps: float = 1e-6) -> pd.Series:
    """Ratio helper that reduces the risk of division by zero."""
    return a / (b.replace(0, np.nan) + eps)


def add_binary_feature(X: pd.DataFrame, name: str, condition: pd.Series) -> None:
    """Convert a boolean condition into a 0/1 feature."""
    X[name] = condition.astype(float).fillna(0).astype(int)


# ------------------------------------------------------------
# 3. Dataset-specific domain feature engineering
# ------------------------------------------------------------
def engineer_neuro_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Meaningful features for the neuroblastoma dataset.

    Clinical rationale:
    - Variables such as risk, stage, MYCN_status, and unfavorable histology are indicators of high risk/prognosis.
    - Follow-up duration in time_months is combined with treatment-related variables.
    """
    X = X.copy()

    age = get_col(X, "age")
    risk = get_col(X, "risk")
    stage = get_col(X, "stage")
    mycn = get_col(X, "MYCN_status")
    histology = get_col(X, "UH_or_FH")
    radiation = get_col(X, "radiation")
    transplant = get_col(X, "autologous_stem_cell_transplantation")
    surgery = get_col(X, "surgical_methods")
    time_months = get_col(X, "time_months")
    differentiation = get_col(X, "degree_of_differentiation")

    if age is not None:
        add_binary_feature(X, "fe_neuro_infant_or_young", age <= 1)
        add_binary_feature(X, "fe_neuro_older_child", age >= 2)

    risk_components = []
    for s in [risk, stage, mycn, histology]:
        if s is not None:
            risk_components.append(s.fillna(0))
    if risk_components:
        X["fe_neuro_composite_risk_score"] = np.vstack(risk_components).sum(axis=0)

    if risk is not None and mycn is not None:
        X["fe_neuro_risk_x_mycn"] = risk.fillna(0) * mycn.fillna(0)

    if radiation is not None and transplant is not None:
        X["fe_neuro_intensive_treatment_count"] = radiation.fillna(0) + transplant.fillna(0)

    if radiation is not None and surgery is not None:
        X["fe_neuro_local_treatment_count"] = radiation.fillna(0) + surgery.fillna(0)

    if time_months is not None:
        X["fe_neuro_log_followup_months"] = np.log1p(time_months.clip(lower=0))
        add_binary_feature(X, "fe_neuro_short_followup", time_months <= time_months.median())

    if differentiation is not None and histology is not None:
        X["fe_neuro_pathology_risk_score"] = differentiation.fillna(0) + histology.fillna(0)

    return X


def engineer_brain_tumor_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Meaningful features for the pediatric brain tumor dataset.

    Clinical rationale:
    - The Age column appears to range between 1992 and 2011 in this file, so it may behave like a birth-year proxy.
      Therefore, fe_brain_birth_year_proxy and relative_age_proxy are used.
    - Symptom burden, tumor aggressiveness, treatment intensity, and dose burden features are created.
    """
    X = X.copy()

    age_year = get_col(X, "Age")
    duration = get_col(X, "Duration of sympthoms before diagnosis (days)")
    icp = get_col(X, "Increased ICP")
    seizure = get_col(X, "Epileptic seizures")
    neuro_deficit = get_col(X, "Neurological deficit")
    hormonal = get_col(X, "Hormonal abnormalities")
    extent = get_col(X, "Extent of disease")
    mri = get_col(X, "MRI finding")
    csf = get_col(X, "CSF cytology")
    radio = get_col(X, "Radiotherapy")
    csi = get_col(X, "CSI dose")
    total_dose = get_col(X, "Total dose")
    concomitant = get_col(X, "Concomitant HT")
    adjuvant = get_col(X, "Adjuvant HT")
    neoadj = get_col(X, "Neoadjuvant HT")

    if age_year is not None:
        X["fe_brain_birth_year_proxy"] = age_year
        X["fe_brain_relative_age_proxy"] = age_year.max() - age_year
        add_binary_feature(X, "fe_brain_older_birth_cohort", age_year <= age_year.median())

    if duration is not None:
        X["fe_brain_log_symptom_duration"] = np.log1p(duration.clip(lower=0))
        add_binary_feature(X, "fe_brain_long_symptom_duration", duration > duration.median())

    symptom_components = []
    for s in [icp, seizure, neuro_deficit, hormonal]:
        if s is not None:
            # Since the dataset is coded as 1/2, value 1 is treated as a positive symptom indicator.
            symptom_components.append((s == 1).astype(int))
    if symptom_components:
        X["fe_brain_symptom_burden"] = np.vstack(symptom_components).sum(axis=0)

    aggressive_components = []
    for col in ["Embryonal tumors", "HGG", "GCT / NGCT", "Ependymoma"]:
        s = get_col(X, col)
        if s is not None:
            aggressive_components.append((s == 1).astype(int))
    if aggressive_components:
        X["fe_brain_aggressive_tumor_count"] = np.vstack(aggressive_components).sum(axis=0)

    if extent is not None and csf is not None:
        X["fe_brain_dissemination_score"] = extent.fillna(0) + (csf == 1).astype(int)

    if mri is not None and extent is not None:
        X["fe_brain_imaging_extent_interaction"] = mri.fillna(0) * extent.fillna(0)

    treatment_components = []
    for s in [radio, concomitant, adjuvant, neoadj]:
        if s is not None:
            treatment_components.append((s == 1).astype(int))
    if treatment_components:
        X["fe_brain_treatment_intensity_count"] = np.vstack(treatment_components).sum(axis=0)

    if csi is not None and total_dose is not None:
        X["fe_brain_total_radiation_burden"] = csi.fillna(0) + total_dose.fillna(0)
        X["fe_brain_csi_total_dose_ratio"] = safe_divide(csi.fillna(0), total_dose.fillna(0))

    return X


def engineer_cancer_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Meaningful features for the colorectal cancer dataset.

    Clinical rationale:
    - Features such as CEA, AJCC, metastasis/progression, lymphovascular/perineural invasion, and signet ring
      are clinically meaningful for cancer prognosis.
    - ASA, comorbidities, and anesthesia duration are indicators of operational/physiological risk.
    """
    X = X.copy()

    age = get_col(X, "age")
    cea = get_col(X, "cea")
    log_cea = get_col(X, "log_cea")
    anes_time = get_col(X, "anes_time")
    interval = get_col(X, "interval")
    interval_r = get_col(X, "interval_r")
    asa = get_col(X, "asa")
    asa3 = get_col(X, "asa3")
    ajcc = get_col(X, "ajcc")
    progress = get_col(X, "progress")
    liver_only = get_col(X, "liver_only")
    cell_diff = get_col(X, "cell_diff")
    rbc = get_col(X, "rbc")

    if age is not None:
        add_binary_feature(X, "fe_crc_age_65_plus", age >= 65)
        add_binary_feature(X, "fe_crc_age_75_plus", age >= 75)
        X["fe_crc_age_group"] = pd.cut(
            age,
            bins=[-np.inf, 50, 65, 75, np.inf],
            labels=[0, 1, 2, 3],
        ).astype(float)

    comorbidity_cols = ["dm", "cad", "hf", "cva", "ckd"]
    existing_comorbidities = [get_col(X, c) for c in comorbidity_cols if get_col(X, c) is not None]
    if existing_comorbidities:
        X["fe_crc_comorbidity_count"] = np.vstack([s.fillna(0) for s in existing_comorbidities]).sum(axis=0)
        add_binary_feature(X, "fe_crc_any_comorbidity", X["fe_crc_comorbidity_count"] > 0)
        add_binary_feature(X, "fe_crc_multi_comorbidity", X["fe_crc_comorbidity_count"] >= 2)

    if asa is not None:
        add_binary_feature(X, "fe_crc_asa_high", asa >= 3)
    if asa is not None and anes_time is not None:
        X["fe_crc_operational_risk_score"] = asa.fillna(0) * np.log1p(anes_time.clip(lower=0))

    if cea is not None:
        X["fe_crc_log1p_cea"] = np.log1p(cea.clip(lower=0))
        add_binary_feature(X, "fe_crc_cea_above_5", cea > 5)
        add_binary_feature(X, "fe_crc_cea_above_median", cea > cea.median())
        add_binary_feature(X, "fe_crc_cea_extreme", cea > cea.quantile(0.90))

    if log_cea is not None and age is not None:
        X["fe_crc_logcea_x_age"] = log_cea.fillna(0) * age.fillna(age.median())

    pathology_cols = [
        "lymphovascularinvasion",
        "perineural",
        "mucin_type",
        "signet_ring",
    ]
    existing_pathology = [get_col(X, c) for c in pathology_cols if get_col(X, c) is not None]
    if existing_pathology:
        X["fe_crc_aggressive_pathology_count"] = np.vstack([s.fillna(0) for s in existing_pathology]).sum(axis=0)
        add_binary_feature(X, "fe_crc_any_aggressive_pathology", X["fe_crc_aggressive_pathology_count"] > 0)

    tumor_progression_components = []
    for s in [ajcc, progress, liver_only, cell_diff]:
        if s is not None:
            tumor_progression_components.append(s.fillna(0))
    if tumor_progression_components:
        X["fe_crc_tumor_progression_score"] = np.vstack(tumor_progression_components).sum(axis=0)

    if anes_time is not None:
        X["fe_crc_log_anesthesia_time"] = np.log1p(anes_time.clip(lower=0))
        add_binary_feature(X, "fe_crc_long_anesthesia", anes_time > anes_time.median())

    if interval is not None and interval_r is not None:
        X["fe_crc_interval_ratio"] = safe_divide(interval.fillna(0), interval_r.fillna(0))
        X["fe_crc_interval_sum"] = interval.fillna(0) + interval_r.fillna(0)
        X["fe_crc_interval_difference"] = interval.fillna(0) - interval_r.fillna(0)

    if rbc is not None and anes_time is not None:
        X["fe_crc_rbc_x_anesthesia"] = rbc.fillna(0) * np.log1p(anes_time.clip(lower=0))

    treatment_cols = ["ct", "rt", "nactrt", "laparoscopic"]
    existing_treatment = [get_col(X, c) for c in treatment_cols if get_col(X, c) is not None]
    if existing_treatment:
        X["fe_crc_treatment_count"] = np.vstack([s.fillna(0) for s in existing_treatment]).sum(axis=0)

    return X


def engineer_sepsis_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Meaningful features for the Sepsis/SIRS dataset.

    Clinical rationale:
    - APACHE II and SOFA are clinical severity scores.
    - CRP, WBCC, neutrophil/lymphocyte balance, and platelet information indicate inflammation/immune response.
    - LOS-ICU may not be post-outcome information and is retained as an indicator of clinical resource utilization during the disease course.
    """
    X = X.copy()

    age = get_col(X, "Age")
    apache = get_col(X, "APACHE II")
    sofa = get_col(X, "SOFA")
    crp = get_col(X, "CRP")
    wbcc = get_col(X, "WBCC")
    neuc = get_col(X, "NeuC")
    lymc = get_col(X, "LymC")
    eoc = get_col(X, "EOC")
    nlcr = get_col(X, "NLCR")
    pltc = get_col(X, "PLTC")
    mpv = get_col(X, "MPV")
    los = get_col(X, "LOS-ICU")

    if age is not None:
        add_binary_feature(X, "fe_sepsis_age_65_plus", age >= 65)
        add_binary_feature(X, "fe_sepsis_age_75_plus", age >= 75)
        X["fe_sepsis_age_group"] = pd.cut(
            age,
            bins=[-np.inf, 40, 65, 75, np.inf],
            labels=[0, 1, 2, 3],
        ).astype(float)

    if apache is not None and sofa is not None:
        X["fe_sepsis_clinical_severity_score"] = apache.fillna(0) + sofa.fillna(0)
        X["fe_sepsis_apache_sofa_interaction"] = apache.fillna(0) * sofa.fillna(0)
        add_binary_feature(X, "fe_sepsis_high_severity", X["fe_sepsis_clinical_severity_score"] > X["fe_sepsis_clinical_severity_score"].median())

    if crp is not None:
        X["fe_sepsis_log_crp"] = np.log1p(crp.clip(lower=0))
    if wbcc is not None:
        X["fe_sepsis_log_wbcc"] = np.log1p(wbcc.clip(lower=0))

    if crp is not None and wbcc is not None:
        X["fe_sepsis_inflammation_score"] = np.log1p(crp.clip(lower=0)) + np.log1p(wbcc.clip(lower=0))

    if neuc is not None and lymc is not None:
        X["fe_sepsis_recalculated_nlr"] = safe_divide(neuc.fillna(0), lymc.fillna(0))
        add_binary_feature(X, "fe_sepsis_lymphopenia", lymc < 1.0)
        add_binary_feature(X, "fe_sepsis_neutrophilia", neuc > neuc.median())

    if nlcr is not None:
        X["fe_sepsis_log_nlcr"] = np.log1p(nlcr.clip(lower=0))
        add_binary_feature(X, "fe_sepsis_high_nlcr", nlcr > nlcr.median())

    if pltc is not None:
        add_binary_feature(X, "fe_sepsis_thrombocytopenia", pltc < 150)
        add_binary_feature(X, "fe_sepsis_high_platelet", pltc > pltc.quantile(0.75))

    if mpv is not None:
        add_binary_feature(X, "fe_sepsis_high_mpv", mpv > mpv.median())

    if pltc is not None and mpv is not None:
        X["fe_sepsis_platelet_mpv_ratio"] = safe_divide(pltc.fillna(0), mpv.fillna(0))

    if eoc is not None:
        add_binary_feature(X, "fe_sepsis_eosinopenia", eoc <= eoc.quantile(0.25))

    if los is not None:
        X["fe_sepsis_log_los_icu"] = np.log1p(los.clip(lower=0))
        add_binary_feature(X, "fe_sepsis_long_icu_stay", los > los.median())

    return X


def engineer_depres_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Meaningful features for the depression + heart failure dataset.

    Clinical rationale:
    - Since Death is the target, "Time from HF to Death (days)" is removed.
    - PHQ-9 represents depression severity, EF represents cardiac pump function, eGFR represents renal function,
      and sodium/BUN and BNP represent heart failure risk.
    """
    X = X.copy()

    leakage_col = "Time from HF to Death (days)"
    if leakage_col in X.columns:
        X = X.drop(columns=[leakage_col])

    age = get_col(X, "Age (years)")
    male = get_col(X, "Male (1=Yes, 0=No)")
    phq = get_col(X, "PHQ-9")
    sbp = get_col(X, "Systolic BP (mm Hg)")
    egfr = get_col(X, "Estimated glomerular filtration rate")
    ef = get_col(X, "Ejection fraction (%)")
    sodium = get_col(X, "Serum sodium (mmol/l)")
    bun = get_col(X, "Blood urea nitrogen (mg/dl)")
    etiology = get_col(X, "Etiology HF(1=Yes, 0=No)")
    diabetes = get_col(X, "Prior diabetes mellitus")
    bnp = get_col(X, "Elevated level of BNP/NT-BNP (1=Yes, 0=No)")
    hosp_time = get_col(X, "Time from HF to hospitalization (days)")
    hospitalized = get_col(X, "Hospitalized (1=Yes, 0=No)")

    if age is not None:
        add_binary_feature(X, "fe_hf_age_65_plus", age >= 65)
        add_binary_feature(X, "fe_hf_age_80_plus", age >= 80)
        X["fe_hf_age_group"] = pd.cut(
            age,
            bins=[-np.inf, 65, 80, np.inf],
            labels=[0, 1, 2],
        ).astype(float)

    if phq is not None:
        add_binary_feature(X, "fe_hf_phq_mild_or_more", phq >= 5)
        add_binary_feature(X, "fe_hf_phq_moderate_or_more", phq >= 10)
        add_binary_feature(X, "fe_hf_phq_severe", phq >= 20)

    if phq is not None and age is not None:
        X["fe_hf_phq_age_interaction"] = phq.fillna(0) * age.fillna(age.median())

    if ef is not None:
        add_binary_feature(X, "fe_hf_reduced_ef", ef < 40)
        add_binary_feature(X, "fe_hf_preserved_ef", ef >= 50)
        X["fe_hf_ef_deficit_from_50"] = (50 - ef).clip(lower=0)

    if egfr is not None:
        add_binary_feature(X, "fe_hf_renal_impairment", egfr < 60)
        add_binary_feature(X, "fe_hf_severe_renal_impairment", egfr < 30)

    if sodium is not None:
        add_binary_feature(X, "fe_hf_hyponatremia", sodium < 135)

    if bun is not None:
        add_binary_feature(X, "fe_hf_high_bun", bun > 20)
        X["fe_hf_log_bun"] = np.log1p(bun.clip(lower=0))

    if bun is not None and egfr is not None:
        X["fe_hf_bun_egfr_ratio"] = safe_divide(bun.fillna(0), egfr.fillna(0))

    risk_components = []
    for s in [diabetes, bnp, hospitalized]:
        if s is not None:
            risk_components.append(s.fillna(0))
    if egfr is not None:
        risk_components.append((egfr < 60).astype(int))
    if ef is not None:
        risk_components.append((ef < 40).astype(int))
    if sodium is not None:
        risk_components.append((sodium < 135).astype(int))
    if phq is not None:
        risk_components.append((phq >= 10).astype(int))
    if risk_components:
        X["fe_hf_multisystem_risk_score"] = np.vstack(risk_components).sum(axis=0)

    if sbp is not None:
        add_binary_feature(X, "fe_hf_low_sbp", sbp < 100)
        add_binary_feature(X, "fe_hf_high_sbp", sbp > 140)

    if hosp_time is not None:
        X["fe_hf_log_time_to_hospitalization"] = np.log1p(hosp_time.clip(lower=0))
        add_binary_feature(X, "fe_hf_early_hospitalization", hosp_time <= hosp_time.median())

    if male is not None and phq is not None:
        X["fe_hf_male_x_phq"] = male.fillna(0) * phq.fillna(0)

    if etiology is not None and bnp is not None:
        X["fe_hf_etiology_x_bnp"] = etiology.fillna(0) * bnp.fillna(0)

    return X


def apply_domain_feature_engineering(X: pd.DataFrame, dataset_name: str) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """
    Apply domain feature engineering based on the dataset name.

    Returns:
    - X_fe: Feature engineered X
    - created_features: newly added columns
    - removed_features: removed columns
    """
    before_cols = set(X.columns)

    if dataset_name == "neuro":
        X_fe = engineer_neuro_features(X)
    elif dataset_name == "braint":
        X_fe = engineer_brain_tumor_features(X)
    elif dataset_name == "cancer":
        X_fe = engineer_cancer_features(X)
    elif dataset_name == "sepsis":
        X_fe = engineer_sepsis_features(X)
    elif dataset_name == "depres":
        X_fe = engineer_depres_features(X)
    else:
        X_fe = X.copy()

    after_cols = set(X_fe.columns)
    created_features = sorted(list(after_cols - before_cols))
    removed_features = sorted(list(before_cols - after_cols))
    return X_fe, created_features, removed_features


# ------------------------------------------------------------
# 4. Dataset audit / data quality assessment function
# ------------------------------------------------------------
def audit_dataset(df: pd.DataFrame, dataset_name: str) -> Dict[str, object]:
    """Return a basic data quality check and feature engineering summary for each dataset."""
    df = clean_column_names(df)
    target_name = get_target_column_name(df, dataset_name)
    X_raw = df.drop(columns=[target_name]).copy()
    X_typed = convert_numeric_like_columns(X_raw)
    X_fe, created_features, removed_domain_features = apply_domain_feature_engineering(X_typed, dataset_name)
    X_clean, removed_duplicate_cols = remove_duplicate_feature_columns(X_fe)

    numeric_cols = X_clean.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_clean.select_dtypes(exclude=np.number).columns.tolist()

    return {
        "dataset": dataset_name,
        "rows": int(df.shape[0]),
        "columns_original": int(df.shape[1]),
        "original_feature_count": int(X_raw.shape[1]),
        "created_domain_features_count": len(created_features),
        "removed_domain_features_count": len(removed_domain_features),
        "feature_columns_after_engineering_and_duplicate_removal": int(X_clean.shape[1]),
        "target_column": target_name,
        "missing_values_total": int(df.isna().sum().sum()),
        "numeric_feature_columns": len(numeric_cols),
        "categorical_feature_columns": len(categorical_cols),
        "created_domain_features": created_features,
        "removed_domain_features": removed_domain_features,
        "removed_duplicate_feature_columns": removed_duplicate_cols,
        "target_distribution": df[target_name].value_counts(dropna=False).to_dict(),
    }


# ------------------------------------------------------------
# 5. X and y preparation function
# ------------------------------------------------------------
def load_dataset(df: pd.DataFrame, dataset_name: str) -> Tuple[pd.DataFrame, np.ndarray, str, Dict[str, object]]:
    """
    Split the dataset into X and y.
    The target variable is selected using the column name defined in TARGET_COLUMN_MAP.
    Apply dataset-specific domain feature engineering.
    """
    df = clean_column_names(df)

    target_name = get_target_column_name(df, dataset_name)
    X = df.drop(columns=[target_name]).copy()
    y_raw = df[target_name].copy()

    X = convert_numeric_like_columns(X)
    X, created_domain_features, removed_domain_features = apply_domain_feature_engineering(X, dataset_name)
    X, removed_duplicate_cols = remove_duplicate_feature_columns(X)

    y = LabelEncoder().fit_transform(y_raw.astype(str))

    if len(np.unique(y)) != 2:
        raise ValueError(
            f"Dataset '{dataset_name}' does not have a binary target. "
            f"Target: {target_name}, classes: {np.unique(y_raw)}"
        )

    numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

    preprocessing_info = {
        "numeric_features": numeric_cols,
        "categorical_features": categorical_cols,
        "removed_duplicate_feature_columns": removed_duplicate_cols,
        "created_domain_features": created_domain_features,
        "removed_domain_features": removed_domain_features,
        "n_numeric_features": len(numeric_cols),
        "n_categorical_features": len(categorical_cols),
        "n_created_domain_features": len(created_domain_features),
        "n_removed_domain_features": len(removed_domain_features),
    }

    print(
        f"Dataset '{dataset_name}' loaded. Target: '{target_name}' | "
        f"Numeric: {len(numeric_cols)} | Categorical: {len(categorical_cols)} | "
        f"Created domain features: {len(created_domain_features)} | "
        f"Removed domain features: {len(removed_domain_features)} | "
        f"Removed duplicate features: {len(removed_duplicate_cols)}"
    )
    return X, y, target_name, preprocessing_info


# ------------------------------------------------------------
# 6. Save engineered datasets
# ------------------------------------------------------------
def create_engineered_dataframe(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """Select the target name using TARGET_COLUMN_MAP; append the target at the end for readability in the saved file."""
    df = clean_column_names(df)
    target_name = get_target_column_name(df, dataset_name)
    X = df.drop(columns=[target_name]).copy()
    y = df[target_name].copy()

    X = convert_numeric_like_columns(X)
    X, _, _ = apply_domain_feature_engineering(X, dataset_name)
    X, _ = remove_duplicate_feature_columns(X)

    engineered_df = X.copy()
    engineered_df[target_name] = y.values
    return engineered_df


def save_engineered_datasets(
    dataframes: Optional[Dict[str, pd.DataFrame]] = None,
    output_dir: Path | str = Path("results_all_experiments/engineered_data"),
) -> Dict[str, str]:
    """
    Save the feature-engineered version of all datasets as CSV files.
    If dataframes are not provided, read them from notebook global variables.
    """
    if dataframes is None:
        dataframes = get_dataframes_from_memory()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = {}
    feature_summary = []
    for dataset_name, df in dataframes.items():
        engineered_df = create_engineered_dataframe(df, dataset_name)
        output_path = output_dir / f"{dataset_name}_engineered.csv"
        engineered_df.to_csv(output_path, index=False)
        saved_paths[dataset_name] = str(output_path)

        original_feature_count = df.shape[1] - 1
        engineered_feature_count = engineered_df.shape[1] - 1
        feature_summary.append({
            "dataset": dataset_name,
            "original_features": original_feature_count,
            "engineered_features": engineered_feature_count,
            "new_features_net": engineered_feature_count - original_feature_count,
            "target_column": target_name if (target_name := get_target_column_name(clean_column_names(df), dataset_name)) else None,
        })

    pd.DataFrame(feature_summary).to_csv(output_dir / "engineered_feature_summary.csv", index=False)
    return saved_paths


# ------------------------------------------------------------
# 7. Build preprocessing pipeline
# ------------------------------------------------------------
def make_one_hot_encoder() -> OneHotEncoder:
    """Create a OneHotEncoder safely across different sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(X: pd.DataFrame, model_name: str) -> ColumnTransformer:
    """
    Build preprocessing according to the model type.
    - Numeric missing value: median imputation
    - Categorical missing value: most_frequent imputation
    - Categorical encoding: OneHotEncoder
    - MinMaxScaler for LR
    """
    model_name = model_name.upper()

    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    if model_name == "LR":
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ])
    else:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    transformers = []
    if numeric_features:
        transformers.append(("num", numeric_pipeline, numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_pipeline, categorical_features))

    if not transformers:
        raise ValueError("No numeric or categorical features are available for the model.")

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )


# ------------------------------------------------------------
# 8. Build model pipeline
# ------------------------------------------------------------
def build_model_pipeline(model_name: str, X: pd.DataFrame) -> Pipeline:
    """Return a preprocessing + model pipeline for RF, LR, or XGBOOST."""
    model_name = model_name.upper()
    preprocessor = build_preprocessor(X, model_name)

    if model_name == "RF":
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=1,
        )

    elif model_name == "LR":
        model = LogisticRegression(
            solver="liblinear",
            max_iter=500,
            class_weight="balanced",
            random_state=42,
        )

    elif model_name == "XGBOOST":
        if XGBClassifier is None:
            raise ImportError("XGBoost not found. Please install it first using: pip install xgboost")
        model = XGBClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            n_jobs=1,
        )

    else:
        raise ValueError("model_name must be one of 'RF', 'LR', 'XGBOOST', or 'CATBOOST'.")

    return Pipeline([
        ("preprocessor", preprocessor),
        ("variance_filter", VarianceThreshold(threshold=0.0)),
        ("model", model),
    ])


# ------------------------------------------------------------
# 9. Repeated hold-out MCC, Accuracy, and ROC AUC evaluation
# ------------------------------------------------------------
def repeated_holdout_metrics(
    X: pd.DataFrame,
    y: np.ndarray,
    model_name: str,
    iterations: int = 100,
    test_size: float = 0.20,
) -> Dict[str, object]:
    """Run repeated hold-out validation for one dataset and one model."""
    mcc_scores: List[float] = []
    accuracy_scores: List[float] = []
    roc_auc_scores: List[float] = []
    start_time = time.perf_counter()

    for i in range(iterations):
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=2026 + i,
            stratify=y,
        )

        pipe = build_model_pipeline(model_name, X_train)
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        mcc = matthews_corrcoef(y_test, y_pred)
        mcc_scores.append(float(mcc))

        accuracy = accuracy_score(y_test, y_pred)
        accuracy_scores.append(float(accuracy))

        try:
            y_pred_proba = pipe.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_pred_proba)
            roc_auc_scores.append(float(roc_auc))
        except AttributeError: # Some models like RandomForestClassifier might not have predict_proba by default if only one class is present
            roc_auc_scores.append(np.nan)
        except ValueError: # Handle cases where only one class is present in y_test, making ROC AUC undefined
            roc_auc_scores.append(np.nan)


    elapsed_seconds = time.perf_counter() - start_time

    return {
        "iterations": iterations,
        "mean_mcc": float(np.mean(mcc_scores)),
        "std_mcc": float(np.std(mcc_scores)),
        "min_mcc": float(np.min(mcc_scores)),
        "max_mcc": float(np.max(mcc_scores)),
        "all_mcc_scores": mcc_scores,
        "mean_accuracy": float(np.mean(accuracy_scores)),
        "std_accuracy": float(np.std(accuracy_scores)),
        "min_accuracy": float(np.min(accuracy_scores)),
        "max_accuracy": float(np.max(accuracy_scores)),
        "all_accuracy_scores": accuracy_scores,
        "mean_roc_auc": float(np.nanmean(roc_auc_scores)) if roc_auc_scores else np.nan,
        "std_roc_auc": float(np.nanstd(roc_auc_scores)) if roc_auc_scores else np.nan,
        "min_roc_auc": float(np.nanmin(roc_auc_scores)) if roc_auc_scores else np.nan,
        "max_roc_auc": float(np.nanmax(roc_auc_scores)) if roc_auc_scores else np.nan,
        "all_roc_auc_scores": roc_auc_scores,
        "elapsed_seconds_dataset_model": float(elapsed_seconds),
    }


# ------------------------------------------------------------
# 10. Function to read CodeCarbon results
# ------------------------------------------------------------
def read_codecarbon_result(emissions_file: Path) -> Dict[str, object]:
    """Read the CodeCarbon emissions CSV and convert kWh to Wh and kg to g."""
    empty = {
        "energy_kWh": np.nan,
        "energy_Wh": np.nan,
        "emissions_kg_CO2eq": np.nan,
        "emissions_g_CO2eq": np.nan,
        "duration_seconds_codecarbon": np.nan,
        "emissions_file": str(emissions_file),
    }
    if not emissions_file.exists():
        return empty

    cc_df = pd.read_csv(emissions_file)
    if cc_df.empty:
        return empty

    last_row = cc_df.iloc[-1]
    energy_kwh = float(last_row.get("energy_consumed", np.nan))
    emissions_kg = float(last_row.get("emissions", np.nan))
    duration_seconds = float(last_row.get("duration", np.nan))

    return {
        "energy_kWh": energy_kwh,
        "energy_Wh": energy_kwh * 1000,
        "emissions_kg_CO2eq": emissions_kg,
        "emissions_g_CO2eq": emissions_kg * 1000,
        "duration_seconds_codecarbon": duration_seconds,
        "emissions_file": str(emissions_file),
    }


# ------------------------------------------------------------
# 11. Main experiment function
# ------------------------------------------------------------
def run_all_experiments(
    iterations: int = 100,
    project_root: Path | str = Path("results_all_experiments"),
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run all experiments.

    Workflow:
    - Engineered datasets are saved as CSV files.
    - A CodeCarbon tracker is started for each model.
    - The same model is run sequentially on the five datasets.
    - MCC mean/std/min/max are calculated for each dataset.
    - The CodeCarbon tracker is stopped after the model finishes.
    """
    if EmissionsTracker is None:
        raise ImportError("CodeCarbon not found. Please install it first using: pip install codecarbon")

    project_root = Path(project_root)
    results_dir = project_root / "results"
    emissions_dir = project_root / "emissions"
    traces_dir = project_root / "mcc_traces"
    engineered_dir = project_root / "engineered_data"

    results_dir.mkdir(parents=True, exist_ok=True)
    emissions_dir.mkdir(parents=True, exist_ok=True)
    traces_dir.mkdir(parents=True, exist_ok=True)
    engineered_dir.mkdir(parents=True, exist_ok=True)

    dataframes = get_dataframes_from_memory()
    model_names = ["RF", "LR", "XGBOOST"]

    all_results = []
    model_energy_results = []
    audits = []

    saved_engineered_paths = save_engineered_datasets(dataframes, engineered_dir)

    for dataset_name, df in dataframes.items():
        audit = audit_dataset(df, dataset_name)
        audit["engineered_csv_path"] = saved_engineered_paths.get(dataset_name)
        audits.append(audit)

    with open(results_dir / "data_audit_with_domain_features.json", "w", encoding="utf-8") as f:
        json.dump(audits, f, indent=2, ensure_ascii=False)

    for model_name in model_names:
        print("\n" + "#" * 70)
        print(f"MODEL STARTED: {model_name}")
        print("#" * 70)

        model_start_time = time.perf_counter()

        emissions_file_name = f"emissions_{model_name}.csv"
        emissions_file_path = emissions_dir / emissions_file_name

        tracker = EmissionsTracker(
            project_name=f"green_computing_{model_name}",
            output_dir=str(emissions_dir),
            output_file=emissions_file_name,
            log_level="error",
        )
        tracker.start()

        model_rows = []

        for dataset_name, df in dataframes.items():
            print("\n" + "=" * 40)
            print(f"Dataset: {dataset_name} | Model: {model_name}")
            print("=" * 40)

            X, y, target_name, preprocessing_info = load_dataset(df, dataset_name)

            metrics = repeated_holdout_metrics(
                X=X,
                y=y,
                model_name=model_name,
                iterations=iterations,
            )

            row = {
                "dataset": dataset_name,
                "model": model_name,
                "target_column": target_name,
                "iterations": metrics["iterations"],
                "mean_mcc": metrics["mean_mcc"],
                "std_mcc": metrics["std_mcc"],
                "min_mcc": metrics["min_mcc"],
                "max_mcc": metrics["max_mcc"],
                "mean_accuracy": metrics["mean_accuracy"],
                "std_accuracy": metrics["std_accuracy"],
                "min_accuracy": metrics["min_accuracy"],
                "max_accuracy": metrics["max_accuracy"],
                "mean_roc_auc": metrics["mean_roc_auc"],
                "std_roc_auc": metrics["std_roc_auc"],
                "min_roc_auc": metrics["min_roc_auc"],
                "max_roc_auc": metrics["max_roc_auc"],
                "elapsed_seconds_dataset_model": metrics["elapsed_seconds_dataset_model"],
                "numeric_features": preprocessing_info["n_numeric_features"],
                "categorical_features": preprocessing_info["n_categorical_features"],
                "created_domain_features": preprocessing_info["n_created_domain_features"],
                "removed_domain_features": preprocessing_info["n_removed_domain_features"],
                "removed_duplicate_features": len(preprocessing_info["removed_duplicate_feature_columns"]),
                "created_domain_feature_names": "; ".join(preprocessing_info["created_domain_features"]),
                "removed_domain_feature_names": "; ".join(preprocessing_info["removed_domain_features"]),
                "preprocessing": (
                    "dataset-specific domain feature engineering; "
                    "numeric: median imputation"
                    + (" + MinMaxScaler" if model_name == "LR" else "")
                    + "; categorical: most_frequent imputation + OneHotEncoder; "
                    + "VarianceThreshold; duplicate feature removal"
                ),
            }

            model_rows.append(row)
            all_results.append(row)

            trace_df = pd.DataFrame({
                "iteration": range(1, iterations + 1),
                "mcc": metrics["all_mcc_scores"],
                "accuracy": metrics["all_accuracy_scores"],
                "roc_auc": metrics["all_roc_auc_scores"],
            })
            trace_df.to_csv(
                traces_dir / f"{dataset_name}_{model_name}_metrics_trace.csv",
                index=False,
            )

        emissions_kg_returned = tracker.stop()
        model_total_seconds = time.perf_counter() - model_start_time

        codecarbon_result = read_codecarbon_result(emissions_file_path)
        codecarbon_result["model"] = model_name
        codecarbon_result["tracker_stop_emissions_kg_CO2eq"] = emissions_kg_returned
        codecarbon_result["model_total_seconds_internal_timer"] = model_total_seconds

        model_energy_results.append(codecarbon_result)

        for row in all_results:
            if row["model"] == model_name:
                row["model_energy_Wh_codecarbon"] = codecarbon_result["energy_Wh"]
                row["model_CO2_g_codecarbon"] = codecarbon_result["emissions_g_CO2eq"]
                row["model_duration_seconds_codecarbon"] = codecarbon_result["duration_seconds_codecarbon"]

        model_table = pd.DataFrame(model_rows)
        model_table["model_energy_Wh_codecarbon"] = codecarbon_result["energy_Wh"]
        model_table["model_CO2_g_codecarbon"] = codecarbon_result["emissions_g_CO2eq"]
        model_table["model_duration_seconds_codecarbon"] = codecarbon_result["duration_seconds_codecarbon"]

        print("\n" + "-" * 70)
        print(f"{model_name} MODEL SUMMARY TABLE")
        print("-" * 70)
        print(
            model_table[
                [
                    "dataset",
                    "model",
                    "mean_mcc",
                    "std_mcc",
                    "mean_accuracy",
                    "std_accuracy",
                    "mean_roc_auc",
                    "std_roc_auc",
                    "created_domain_features",
                    "numeric_features",
                    "categorical_features",
                    "model_energy_Wh_codecarbon",
                    "model_CO2_g_codecarbon",
                    "model_duration_seconds_codecarbon",
                ]
            ].to_string(index=False)
        )

    final_results_df = pd.DataFrame(all_results)
    model_energy_df = pd.DataFrame(model_energy_results)

    final_columns = [
        "dataset",
        "model",
        "target_column",
        "iterations",
        "mean_mcc",
        "std_mcc",
        "min_mcc",
        "max_mcc",
        "mean_accuracy",
        "std_accuracy",
        "min_accuracy",
        "max_accuracy",
        "mean_roc_auc",
        "std_roc_auc",
        "min_roc_auc",
        "max_roc_auc",
        "numeric_features",
        "categorical_features",
        "created_domain_features",
        "removed_domain_features",
        "removed_duplicate_features",
        "elapsed_seconds_dataset_model",
        "model_energy_Wh_codecarbon",
        "model_CO2_g_codecarbon",
        "model_duration_seconds_codecarbon",
        "created_domain_feature_names",
        "removed_domain_feature_names",
        "preprocessing",
    ]
    final_results_df = final_results_df[final_columns]

    final_results_df.to_csv(
        results_dir / "final_results_all_datasets_all_models_domain_features.csv",
        index=False,
    )
    model_energy_df.to_csv(
        results_dir / "model_level_codecarbon_summary_domain_features.csv",
        index=False,
    )

    print("\n" + "#" * 70)
    print("FINAL RESULTS - ALL DATASETS AND ALL MODELS")
    print("#" * 70)
    print(final_results_df.to_string(index=False))

    print("\n" + "#" * 70)
    print("MODEL LEVEL CODECARBON SUMMARY")
    print("#" * 70)
    print(
        model_energy_df[
            [
                "model",
                "energy_Wh",
                "emissions_g_CO2eq",
                "duration_seconds_codecarbon",
                "model_total_seconds_internal_timer",
            ]
        ].to_string(index=False)
    )

    print(f"\nResult files saved to: {results_dir}")
    print(f"Feature-engineered CSV files saved to: {engineered_dir}")
    return final_results_df, model_energy_df


# ------------------------------------------------------------
# 12. Script entry point
# ------------------------------------------------------------
if __name__ == "__main__":
    final_results_df, model_energy_df = run_all_experiments(
        iterations=100,
        project_root=Path("results_all_experiments"),
    )

sorted_results = final_results_df.sort_values(by=['mean_mcc', 'mean_roc_auc'], ascending=False)

display(sorted_results[['dataset', 'model', 'mean_mcc', 'std_mcc', 'mean_accuracy', 'std_accuracy', 'mean_roc_auc', 'std_roc_auc']])


######################################################################
MODEL STARTED: RF
######################################################################

Dataset: neuro | Model: RF
Dataset 'neuro' loaded. Target: 'outcome' | Numeric: 21 | Categorical: 0 | Created domain features: 9 | Removed domain features: 0 | Removed duplicate features: 0

Dataset: braint | Model: RF
Dataset 'braint' loaded. Target: 'Status OS' | Numeric: 37 | Categorical: 0 | Created domain features: 12 | Removed domain features: 0 | Removed duplicate features: 5

Dataset: cancer | Model: RF
Dataset 'cancer' loaded. Target: 'death' | Numeric: 53 | Categorical: 0 | Created domain features: 23 | Removed domain features: 0 | Removed duplicate features: 1

Dataset: sepsis | Model: RF
Dataset 'sepsis' loaded. Target: 'Mortality' | Numeric: 36 | Categorical: 0 | Created domain features: 21 | Removed domain features: 0 | Removed duplicate features: 0

Dataset: depres | Model: RF
Dataset 'depres' loaded. Target: 'D

,dataset,model,mean_mcc,std_mcc,mean_accuracy,std_accuracy,mean_roc_auc,std_roc_auc
11,braint,XGBOOST,0.828227,0.081010,0.912571,0.040772,0.933882,0.046612
1,braint,RF,0.781720,0.088141,0.889143,0.044314,0.922911,0.049774
6,braint,LR,0.692629,0.112777,0.845143,0.056015,0.913553,0.043853
8,sepsis,LR,0.571618,0.053339,0.877302,0.019703,0.928683,0.023527
13,sepsis,XGBOOST,0.536304,0.080601,0.920278,0.012650,0.931891,0.023641
10,neuro,XGBOOST,0.528301,0.149697,0.763824,0.073642,0.826982,0.072491
3,sepsis,RF,0.524884,0.086799,0.922579,0.012655,0.929759,0.024458
5,neuro,LR,0.494907,0.136533,0.743824,0.070009,0.800684,0.074218
0,neuro,RF,0.465606,0.140713,0.731176,0.070835,0.784140,0.068730
14,depres,XGBOOST,0.422048,0.101969,0.813176,0.029961,0.833862,0.044556


In [ ]:
sorted_results = final_results_df.sort_values(by=['mean_mcc', 'mean_roc_auc'], ascending=False)

display(sorted_results[['dataset', 'model', 'mean_mcc', 'std_mcc', 'mean_accuracy', 'std_accuracy', 'mean_roc_auc', 'std_roc_auc']])